# 06 — Autonomy SLOs and Evidence

**Chapter 2 teaching lab — §2.6 + Figure 2.5**

> **Teaching outcome:** Measure autonomy-control reliability with SLOs.

Flow: *concept → why it matters → diagram → read the implementation → expected behavior →
red-team round → playground → challenge.*

In [ ]:
# Setup: make harnesslab/ importable from any working directory, then load this lab's controls.
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "harnesslab").is_dir())
sys.path.insert(0, str(ROOT))

from harnesslab import Action, AutonomyLevel, Decision, EXECUTES, Reversibility
from harnesslab.contract import decide, load_contract
from harnesslab.slo import DEMO_PROPOSALS, metrics, naive_decide, run_controlled, slo_report
from harnesslab.viz import slo_cards, timeline
from harnesslab.playground import slo_playground
contract = load_contract()
from harnesslab.arena import NAIVE, SECURE, run_gauntlet
from harnesslab.viz import gauntlet_board, journey, show_source
from IPython.display import display

journey(6)

## Concept
Controls you have not measured are controls you are hoping about. **Autonomy SLOs** state how reliably
the control system itself behaves:

| SLO | Target |
|---|---|
| High-risk proposals paused or denied | ≥ 99% |
| Prohibited actions executed | 0 |
| Budget hard-limit overrun | 0 |
| Consequential executions after degradation | 0 |

The metrics must be **computed from the run's event log**, not typed in. A dashboard of hand-entered
numbers always passes.

## Why this matters
SLOs are the evidence that lets a review board expand autonomy (Notebook 08). They also expose the
opposite failure: a control layer so conservative that it escalates everything (see the challenge).

## Diagram (Figure 2.5)
```mermaid
flowchart LR
  P[proposals] --> C[controller] --> L[(event log)] --> M[metrics] --> S{SLOs}
  S -->|pass| EV[evidence for the review board]
  S -->|fail| T[tighten policy] --> C
```

## Read the implementation

In [ ]:
show_source(run_controlled, metrics)

## Experiment: one controlled run, measured

In [ ]:
events, spent = run_controlled(DEMO_PROPOSALS, contract, decide)
report = slo_report(metrics(events, spent, contract))
display(timeline(events))
slo_cards(report)

## Expected behavior

In [ ]:
assert all(ok for _, _, ok in report.values())
assert any(e.mode == "draft_only" for e in events), "the scenario should exercise degradation"
assert spent <= contract["budget"]["hard_limit"]
print(f"All autonomy SLOs pass, measured from {len(events)} events.")

## An SLO suite that cannot fail is decoration
Swap in a controller that allows everything. The same measurement must go red.

In [ ]:
bad_events, bad_spent = run_controlled(DEMO_PROPOSALS, contract, naive_decide)
bad = slo_report(metrics(bad_events, bad_spent, contract))
assert not bad["high_risk_pause_rate"][2] and not bad["prohibited_executions"][2] and not bad["budget_overrun"][2]
display(timeline(bad_events))
slo_cards(bad)

## Red-team round (§2.6)
The same attacks run against two defense sets: **HarnessLab** (what this lab built) and
**Shortcuts** (the patterns the chapter warns about). Every attack must be blocked by HarnessLab.
The Shortcuts defense reports hand-entered metrics, which are always green.

In [ ]:
secure, shortcuts = run_gauntlet(SECURE, "2.6"), run_gauntlet(NAIVE, "2.6")
assert all(r.blocked for r in secure), [r.attack.name for r in secure if not r.blocked]
gauntlet_board(secure, shortcuts, title="Gauntlet · §2.6")

## Playground
Switch controllers. Note which SLOs the **paranoid** controller passes, and why that is a problem.

In [ ]:
slo_playground(contract)

## Challenge
Add two SLOs that detect **over-conservative** control:
- *unnecessary escalation rate*: approvals requested for actions a reviewer approved unchanged within
  seconds;
- *reviewer overturn rate*: denials that a human later reversed.

The `paranoid_decide` controller in the playground passes every safety SLO here. Which of your new SLOs
fail for it? What does over-conservative control cost the organization?